## Homework 9: Text Classification with Fine-Tuned BERT

In this final homework, we’ll explore **fine-tuning a pre-trained Transformer model (BERT)** for text classification using the **IMDB Movie Review** dataset. You’ll begin with a working baseline notebook and then conduct a series of controlled experiments to understand how data size, context length, and model architecture affect performance.

You’ll complete three problems:

* **Problem 1:** Evaluate how **sequence length** and **learning rate** jointly influence validation loss and generalization.
* **Problem 2:** Measure how **training data size** affects both model performance and total training time.
* **Problem 3:** Compare **two additional models** from the BERT family to analyze the trade-offs between model size and accuracy on this dataset.

In each problem, you’ll report your key metrics, summarize what you observed, and reflect on what you learned.

> **Note:** This homework was developed and tested on **Google Colab**, due to version conflicts when running locally. It is **strongly recommended** that you complete your work on Colab as well.

There are 6 problems, each worth 14 points, and you get one point free if you complete the entire homework.


In [1]:
# Install once per new Colab runtime
#%pip -q install -U keras keras-hub tensorflow tensorflow-text datasets evaluate
%pip install keras keras-hub tensorflow-text datasets evaluate
%pip install tensorflow-text==2.19.*

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.4 MB/s eta 0:00:00


In [2]:

import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import time
import random
import numpy as np
import keras
import keras_hub as kh
import evaluate
from datasets import load_dataset, Dataset, Features, Value, ClassLabel

from keras import mixed_precision                    # generally faster
mixed_precision.set_global_policy("mixed_float16")

### Here is where you can set global hyperparameters for this homework

In [3]:
# ---------------- Config ----------------
SEED        = 42
MAX_LEN     = 128
EPOCHS      = 3
BATCH       = 32
EVAL_BATCH  = 64
SUBSET_FRAC = 0.25   # <-- 0.25 to train and test on 25% of whole dataset during development;  set to 1.0 for full dataset

keras.utils.set_random_seed(SEED)

### Load and Preprocess the IMDB Movie Review Dataset

In [4]:
# ---- Load IMDb (raw), join train+test ----
imdb   = load_dataset("imdb")
texts  = list(imdb["train"]["text"]) + list(imdb["test"]["text"])
labels = np.array(list(imdb["train"]["label"]) + list(imdb["test"]["label"]), dtype="int32")

# ---- Build DS with explicit features (label=ClassLabel) ----
features = Features({"text": Value("string"),
                     "label": ClassLabel(num_classes=2, names=["NEG","POS"])})
all_ds = Dataset.from_dict({"text": texts, "label": labels.tolist()}, features=features)

# ---- Optional: take a stratified subset of the FULL dataset ----
if 0.0 < SUBSET_FRAC < 1.0:
    sub = all_ds.train_test_split(train_size=SUBSET_FRAC, seed=SEED, stratify_by_column="label")
    ds_pool = sub["train"]
else:
    ds_pool = all_ds

# ---- Stratified 80/10/10 split on the (possibly smaller) pool ----
# First: 80/20 train+val pool / test
splits = ds_pool.train_test_split(test_size=0.20, seed=SEED, stratify_by_column="label")
train_val_pool, test_ds = splits["train"], splits["test"]
# Then: carve 10% of full (i.e., 0.125 of the 80% pool) as validation
splits2 = train_val_pool.train_test_split(test_size=0.125, seed=SEED, stratify_by_column="label")
train_ds, val_ds = splits2["train"], splits2["test"]

# ---- Numpy arrays for Keras fit/predict ----
X_tr = np.array(train_ds["text"], dtype=object); y_tr = np.array(train_ds["label"], dtype="int32")
X_va = np.array(val_ds["text"],   dtype=object); y_va = np.array(val_ds["label"],   dtype="int32")
X_te = np.array(test_ds["text"],  dtype=object); y_te = np.array(test_ds["label"],  dtype="int32")

# ---- Quick summary ----
def _counts(ds):
    arr = np.array(ds["label"], dtype=int)
    return len(arr), np.bincount(arr, minlength=2).tolist()
print(f"Pool after SUBSET_FRAC={SUBSET_FRAC}: {len(ds_pool)} (of {len(all_ds)})")
print("Train:", _counts(train_ds), " Val:", _counts(val_ds), " Test:", _counts(test_ds))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Pool after SUBSET_FRAC=0.25: 12500 (of 50000)
Train: (8750, [4375, 4375])  Val: (1250, [625, 625])  Test: (2500, [1250, 1250])


### Build and train a baseline Distil-Bert Text Classifier

In [ ]:
# ---- Keras Hub preprocessor + classifier ----
preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
    "distil_bert_base_en_uncased", sequence_length=MAX_LEN
)
model = kh.models.DistilBertTextClassifier.from_preset(
    "distil_bert_base_en_uncased", num_classes=2, preprocessor=preproc
)

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
)

start = time.time()

# ---- Train with early stopping (restore best val weights) ----
cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)]
history = model.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=EPOCHS,
    batch_size=BATCH,
    callbacks=cb,
    verbose=1,
)

# ---- Evaluate (accuracy + F1 via `evaluate`) ----
logits = model.predict(X_te, batch_size=EVAL_BATCH, verbose=0)
y_pred = logits.argmax(axis=-1)

acc_metric = evaluate.load("accuracy")
f1_metric  = evaluate.load("f1")
acc = acc_metric.compute(predictions=y_pred, references=y_te)["accuracy"]
f1  = f1_metric.compute(predictions=y_pred, references=y_te)["f1"]

# Tiny confusion matrix helper (no sklearn needed)
def confusion_matrix_np(y_true, y_pred, num_classes=2):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

print(f"\nValidation acc (best epoch): {history.history['val_acc'][np.argmin(history.history['val_loss'])]:.3f}")
print(f"\nTest accuracy: {acc:.3f}   Test F1: {f1:.3f}")
print("\nConfusion matrix:\n", confusion_matrix_np(y_te, y_pred))

end = time.time() - start
print("\nElapsed time:", time.strftime("%H:%M:%S", time.gmtime(end)))

Epoch 1/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 104s 226ms/step - acc: 0.7824 - loss: 0.4530 - val_acc: 0.8376 - val_loss: 0.3448
Epoch 2/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 10s 36ms/step - acc: 0.8787 - loss: 0.2897 - val_acc: 0.8584 - val_loss: 0.3400
Epoch 3/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 10s 35ms/step - acc: 0.9160 - loss: 0.2207 - val_acc: 0.8592 - val_loss: 0.3555

Validation acc (best epoch): 0.858

Test accuracy: 0.855   Test F1: 0.851

Confusion matrix:
 [[1098  152]
 [ 211 1039]]

Elapsed time: 00:02:24


# Problem 1 — Mini sweep: context length × learning rate (6 runs)

In this problem we'll see how much **context length** (`MAX_LEN`) helps, and how sensitive fine-tuning is to **learning rate**—without running a huge grid.

## Setup (keep these fixed)

* `SUBSET_FRAC = 0.25`               # use only this percentage of the whole dataset
* `EPOCHS = 3`
* `BATCH = 32` (but see note for 256 below)
* **EarlyStopping** with `restore_best_weights=True`
* Same random `SEED` for all runs
* Same data split for all runs (don’t reshuffle between runs)

### Run these 6 configurations

**For each** `MAX_LEN ∈ {128, 256, 512}`, try **two** learning rates:

* **MAX_LEN = 128**

  * `(LR = 2e-5, BATCH = 32)` – healthy default for shorter contexts.
  * `(LR = 1e-5, BATCH = 32)` – conservative LR; often a touch stabler.

* **MAX_LEN = 256**

  * `(LR = 1e-5, BATCH = 16)` – longer context → lower batch.
  * `(LR = 7.5e-6, BATCH = 16)` – even steadier if loss is noisy.

* **MAX_LEN = 512**  *(heavier quadratic attention cost)*

  * `(LR = 7.5e-6, BATCH = 8)` – safe starting point.
  * `(LR = 5e-6, BATCH = 8)` – extra caution for stability.

**If you hit an Out Of Memory error:**

* At **256** with `BATCH = 16`, drop to `BATCH = 8`.
* At **512** with `BATCH = 8`, drop to `BATCH = 4`.


Then answer the graded questions.


In [ ]:
# Your code here; add as many cells as you need
## set up
acc_metric = evaluate.load("accuracy")
f1_metric  = evaluate.load("f1")

def confusion_matrix_np(y_true, y_pred, num_classes=2):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

def p1_run_experiment(run_name, max_len, lr, batch):
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)

    start = time.time()

    preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
        "distil_bert_base_en_uncased",
        sequence_length=max_len
    )

    model = kh.models.DistilBertTextClassifier.from_preset(
        "distil_bert_base_en_uncased",
        num_classes=2,
        preprocessor=preproc
    )

    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
    )

    cb = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=2,
            restore_best_weights=True
        )
    ]

    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_va, y_va),
        epochs=EPOCHS,
        batch_size=batch,
        callbacks=cb,
        verbose=1,
    )

    logits = model.predict(X_te, batch_size=EVAL_BATCH, verbose=0)
    y_pred = logits.argmax(axis=-1)

    acc = acc_metric.compute(predictions=y_pred, references=y_te)["accuracy"]
    f1  = f1_metric.compute(predictions=y_pred, references=y_te)["f1"]

    best_idx      = int(np.argmin(history.history["val_loss"]))
    best_epoch    = best_idx + 1
    best_val_loss = float(history.history["val_loss"][best_idx])
    best_val_acc  = float(history.history["val_acc"][best_idx])

    elapsed = time.time() - start

    print(f"\n{run_name}")
    print(f"Best epoch: {best_epoch}")
    print(f"Validation acc (best epoch): {best_val_acc:.3f}")
    print(f"Validation loss (best epoch): {best_val_loss:.3f}")
    print(f"Test accuracy: {acc:.3f}")
    print(f"Test F1: {f1:.3f}")
    print("\nConfusion matrix:\n", confusion_matrix_np(y_te, y_pred))
    print("\nElapsed time:", time.strftime("%H:%M:%S", time.gmtime(elapsed)))

    return {
        "run_name": run_name,
        "MAX_LEN": max_len,
        "LR": lr,
        "BATCH": batch,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "best_val_acc": best_val_acc,
        "test_acc": acc,
        "test_f1": f1,
        "elapsed_sec": elapsed,
    }

In [ ]:
## prob 1 run 1
p1_run1 = p1_run_experiment(
    run_name="Run 1: MAX_LEN=128, LR=2e-5, BATCH=32",
    max_len=128,
    lr=2e-5,
    batch=32
)

In [ ]:
## prob 1 run 2
p2_run2 = p1_run_experiment(
    run_name="Run 2: MAX_LEN=128, LR=1e-5, BATCH=32",
    max_len=128,
    lr=1e-5,
    batch=32
)

In [ ]:
## prob 1 run 3
p3_run3 = p1_run_experiment(
    run_name="Run 3: MAX_LEN=256, LR=1e-5, BATCH=16",
    max_len=256,
    lr=1e-5,
    batch=16
)

In [ ]:
## prob 1 run 4
p1_run4 = p1_run_experiment(
    run_name="Run 4: MAX_LEN=256, LR=7.5e-6, BATCH=8",
    max_len=256,
    lr=7.5e-6,
    batch=8
)

In [ ]:
## prob 1 run 5
p1_run5 = p1_run_experiment(
    run_name="Run 5: MAX_LEN=512, LR=7.5e-6, BATCH=4",
    max_len=512,
    lr=7.5e-6,
    batch=4
)

In [ ]:
## prob 1 run 6
p1_run6 = p1_run_experiment(
    run_name="Run 6: MAX_LEN=512, LR=5e-6, BATCH=4",
    max_len=512,
    lr=5e-6,
    batch=4
)

In [ ]:
## compare
p1_results_df = pd.DataFrame([p1_run1, p1_run2, p1_run3, p1_run4, p1_run5, p1_run6])
p1_results_df = p1_results_df.sort_values(
    by=["test_f1", "test_acc", "best_val_acc"],
    ascending=False
).reset_index(drop=True)

display(p1_results_df)

### Graded Questions

In [ ]:
# Set a1a to the validation accuracy at min validation loss for your best configuration found in this problem

a1a = 0.0             # Replace 0.0 with your answer

In [ ]:
# Graded Answer
# DO NOT change this cell in any way

print(f'a1a = {a1a:.4f}')

a1a = 0.0000


#### Question a1b:

* Does **more context** (128 → 256 → 512) consistently help?
* How much effect did the learning rate have on the validation accuracy?


#### Your Answer Here:

## Problem 2 — How much data is enough?

In this problem, you’ll investigate how model performance scales with dataset size.

**Setup.**
Use the best `MAX_LEN` and `LR` values you found in **Problem 1**.

**What to do:**

1. For each value of `SUBSET_FRAC ∈ {0.25, 0.50, 0.75, 1.00}`, train your model once and observe the displayed performance metrics.
2. Answer the discussion question below.




In [ ]:
# Your code here; add as many cells as you need
acc_metric = evaluate.load("accuracy")
f1_metric  = evaluate.load("f1")

def confusion_matrix_np(y_true, y_pred, num_classes=2):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

def p2_run_experiment(df, subset_frac, max_len, lr, batch):
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)

    print("\n" + "="*80)
    print(f"SUBSET_FRAC = {subset_frac}")
    print("="*80)

    # ---- Create subset ----
    df_subset = df.sample(frac=subset_frac, random_state=SEED).reset_index(drop=True)

    # ---- SAME split logic every time ----
    train_df, temp_df = train_test_split(
        df_subset,
        test_size=0.30,
        random_state=SEED,
        stratify=df_subset["label"]
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=SEED,
        stratify=temp_df["label"]
    )

    X_tr, y_tr = train_df["text"], train_df["label"]
    X_va, y_va = val_df["text"], val_df["label"]
    X_te, y_te = test_df["text"], test_df["label"]

    start = time.time()

    # ---- Model ----
    preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
        "distil_bert_base_en_uncased",
        sequence_length=max_len
    )

    model = kh.models.DistilBertTextClassifier.from_preset(
        "distil_bert_base_en_uncased",
        num_classes=2,
        preprocessor=preproc
    )

    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
    )

    cb = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=2,
            restore_best_weights=True
        )
    ]

    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_va, y_va),
        epochs=EPOCHS,
        batch_size=batch,
        callbacks=cb,
        verbose=1,
    )

    # ---- Evaluate ----
    logits = model.predict(X_te, batch_size=EVAL_BATCH, verbose=0)
    y_pred = logits.argmax(axis=-1)

    acc = acc_metric.compute(predictions=y_pred, references=y_te)["accuracy"]
    f1  = f1_metric.compute(predictions=y_pred, references=y_te)["f1"]

    best_idx = int(np.argmin(history.history["val_loss"]))

    print(f"\nValidation acc (best epoch): {history.history['val_acc'][best_idx]:.3f}")
    print(f"Test accuracy: {acc:.3f}   Test F1: {f1:.3f}")
    print("\nConfusion matrix:\n", confusion_matrix_np(y_te, y_pred))

    elapsed = time.time() - start
    print("\nElapsed time:", time.strftime("%H:%M:%S", time.gmtime(elapsed)))

    return {
        "subset_frac": subset_frac,
        "val_acc": history.history['val_acc'][best_idx],
        "test_acc": acc,
        "test_f1": f1,
        "time_sec": elapsed
    }

In [ ]:
## p2 run 1 (25%)
p2_run1 = p2_run_experiment(df, 0.25, BEST_MAX_LEN, BEST_LR, BATCH)

In [ ]:
## p2 run 2 (50%)
p2_run2 = p2_run_experiment(df, 0.50, BEST_MAX_LEN, BEST_LR, BATCH)

In [ ]:
## p2 run 3 (75%)
p2_run3 = p2_run_experiment(df, 0.75, BEST_MAX_LEN, BEST_LR, BATCH)

In [ ]:
## p2 run 4 (100%)
p2_run4 = p2_run_experiment(df, 1.00, BEST_MAX_LEN, BEST_LR, BATCH)

In [ ]:
## compare
p2_results_df = pd.DataFrame([p2_run1, p2_run2, p2_run3, p2_run4])
display(results_df)

### Graded Questions

In [ ]:
# Set a2a to the validation accuracy at min validation loss for your best configuration found in this problem
# (Yes, it is probably at 1.0!)

a2a = 0.0             # Replace 0.0 with your answer

In [ ]:
# Graded Answer
# DO NOT change this cell in any way

print(f'a2a = {a2a:.4f}')

a2a = 0.0000


#### Question a2b:

Summarize what you observed as dataset size increased. Given that validation metrics are typically reliable to only about two decimal places, do the performance gains justify using the entire dataset? What trade-offs between accuracy and computation time did you notice?

#### Your Answer Here:

# Problem 3 — Model swap: speed vs. accuracy (why: capacity matters)

In this problem we will compare encoder-only backbones of different sizes.

**Setup.** Keep the best `MAX_LEN`, `LR`, and `SUBSET_FRAC` from Problems 1–2. Only change the model/preset:

* **DistilBERT** (current baseline)
* **BERT-base** (larger/usually stronger)

**How to switch (two lines each).**

* DistilBERT:

  ```python
  preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset("distil_bert_base_en_uncased", sequence_length=MAX_LEN)
  model  = kh.models.DistilBertTextClassifier.from_preset("distil_bert_base_en_uncased", num_classes=2, preprocessor=preproc)
  ```

* BERT-base:

  ```python
  preproc = kh.models.BertTextClassifierPreprocessor.from_preset("bert_base_en_uncased", sequence_length=MAX_LEN)
  model  = kh.models.BertTextClassifier.from_preset("bert_base_en_uncased", num_classes=2, preprocessor=preproc)
  ```

**What to do.**

1. Train/evaluate each model once with identical settings.
2. Observe the performance metrics for each.
3. Answer the graded questions.



In [ ]:
# Your code here; add as many cells as you wish
# Fixed subset + fixed split ONCE
df_subset = all_ds.sample(frac=SUBSET_FRAC, random_state=SEED).reset_index(drop=True)

train_df, temp_df = train_test_split(
    df_subset,
    test_size=0.30,
    random_state=SEED,
    stratify=df_subset["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

X_tr, y_tr = train_df["text"], train_df["label"]
X_va, y_va = val_df["text"], val_df["label"]
X_te, y_te = test_df["text"], test_df["label"]

print("Train size:", len(X_tr))
print("Val size:  ", len(X_va))
print("Test size: ", len(X_te))

def run_model_swap(run_name, preproc_class, model_class, preset_name):
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)

    start = time.time()

    preproc = preproc_class.from_preset(
        preset_name,
        sequence_length=MAX_LEN
    )

    model = model_class.from_preset(
        preset_name,
        num_classes=2,
        preprocessor=preproc
    )

    model.compile(
        optimizer=keras.optimizers.Adam(LR),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
    )

    cb = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=2,
            restore_best_weights=True
        )
    ]

    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_va, y_va),
        epochs=EPOCHS,
        batch_size=BATCH,
        callbacks=cb,
        verbose=1,
    )

    logits = model.predict(X_te, batch_size=EVAL_BATCH, verbose=0)
    y_pred = logits.argmax(axis=-1)

    acc = acc_metric.compute(predictions=y_pred, references=y_te)["accuracy"]
    f1  = f1_metric.compute(predictions=y_pred, references=y_te)["f1"]

    best_idx      = int(np.argmin(history.history["val_loss"]))
    best_epoch    = best_idx + 1
    best_val_loss = float(history.history["val_loss"][best_idx])
    best_val_acc  = float(history.history["val_acc"][best_idx])

    elapsed = time.time() - start

    print("\n" + "=" * 80)
    print(run_name)
    print("=" * 80)
    print(f"Best epoch: {best_epoch}")
    print(f"Validation acc (best epoch): {best_val_acc:.3f}")
    print(f"Validation loss (best epoch): {best_val_loss:.3f}")
    print(f"Test accuracy: {acc:.3f}")
    print(f"Test F1: {f1:.3f}")
    print("\nConfusion matrix:\n", confusion_matrix_np(y_te, y_pred))
    print("\nElapsed time:", time.strftime("%H:%M:%S", time.gmtime(elapsed)))

    return {
        "model": run_name,
        "MAX_LEN": MAX_LEN,
        "LR": LR,
        "SUBSET_FRAC": SUBSET_FRAC,
        "BATCH": BATCH,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "best_val_acc": best_val_acc,
        "test_acc": acc,
        "test_f1": f1,
        "elapsed_sec": elapsed,
    }

In [ ]:
## p3 run 1 (DistilBERT run)
distilbert_result = run_model_swap(
    run_name="DistilBERT",
    preproc_class=kh.models.DistilBertTextClassifierPreprocessor,
    model_class=kh.models.DistilBertTextClassifier,
    preset_name="distil_bert_base_en_uncased"
)

In [ ]:
## p3 run 2 (BERT-base run)
bert_result = run_model_swap(
    run_name="BERT-base",
    preproc_class=kh.models.BertTextClassifierPreprocessor,
    model_class=kh.models.BertTextClassifier,
    preset_name="bert_base_en_uncased"
)

In [ ]:
## compare
p3_results_df = pd.DataFrame([distilbert_result, bert_result])
p3_results_df = p3_results_df.sort_values(
    by=["test_f1", "test_acc", "best_val_acc"],
    ascending=False
).reset_index(drop=True)

display(p3_results_df)

### Graded Questions

In [ ]:
# Set a1a to the validation accuracy at min validation loss for your best model found in this problem

a3a = 0.0             # Replace 0.0 with your answer

In [ ]:
# Graded Answer
# DO NOT change this cell in any way

print(f'a3a = {a3a:.4f}')

a3a = 0.0000


#### Question a3b:

**Answer briefly.**

* Which model gives the best **accuracy/F1**?
* Which is **fastest** per epoch?
* Given limited development time or compute resources, which model is the best **overall choice** and why?

#### Your Answer Here: